# Business insights from incrementality

This notebook summarizes **experimental incrementality** for retail media campaigns: how much extra conversion and revenue the **treatment** group drove versus a **control** group, holding the campaign window fixed.

**How to use this:** Treat these tables as inputs to budget and planning conversations—where to **scale**, **hold**, **watch**, or **pull back** spend—not as the only signals (creative, inventory, and seasonality still matter).

---

In [6]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

# Project root: cwd, or parent when the kernel runs from ``notebooks/``
ROOT = Path.cwd()
if not (ROOT / "app").is_dir() and (ROOT.parent / "app").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.core.database import get_engine

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

engine = get_engine()

lift = pd.read_sql_table("experiment_lift_metrics", engine, schema="marts")
rankings = pd.read_sql_table("campaign_incrementality_rankings", engine, schema="marts")
flags = pd.read_sql_table("campaign_efficiency_flags", engine, schema="marts")

print(f"Loaded rows: experiment_lift_metrics={len(lift)}, rankings={len(rankings)}, efficiency_flags={len(flags)}")

Loaded rows: experiment_lift_metrics=24, rankings=24, efficiency_flags=24


### Insights to keep in mind

1. **Incremental revenue** estimates *additional* revenue from the treatment arm versus what you would expect if treatment behaved like control—useful for **sizing financial upside**, not for attribution to a single touchpoint.
2. **Absolute lift** (treatment minus control conversion rate) answers whether the campaign **moved the needle on conversion**; large positive lift supports **defending or growing** investment when revenue aligns.
3. **Rankings** (`campaign_incrementality_rankings`) combine normalized lift and revenue into a **single prioritization score**—helpful when leadership wants a short list, not a spreadsheet debate.
4. **Efficiency flags** bucket campaigns into coarse tiers (`high_impact`, `moderate`, `low_impact`, `inefficient`); pair them with **incremental revenue** so a “high impact” label is not mistaken for **material dollar impact**.
5. **Negative lift** means treatment converted **worse than control** in the experiment—before cutting budget, confirm data quality and whether the test window matches **live** planning; still, these are **first candidates** for **hold tests, creative swaps, or reduced spend**.

---

### Top 5 campaigns by incremental revenue

Higher **incremental_revenue** means more estimated **incremental dollars** tied to the treatment cohort in this experiment (see mart definitions for the exact formula).

In [7]:
cols = ["campaign_id", "incremental_revenue", "absolute_lift", "treatment_conversion_rate", "control_conversion_rate"]
top_rev = lift.sort_values("incremental_revenue", ascending=False, na_position="last").head(5)[cols]
top_rev

,campaign_id,incremental_revenue,absolute_lift,treatment_conversion_rate,control_conversion_rate
11,6,671.64,0.411765,0.411765,0.000000
18,24,525.30,0.229167,0.229167,0.000000
2,9,468.83,0.218182,0.218182,0.000000
3,15,458.26,0.299283,0.354839,0.055556
7,17,415.39,0.247863,0.358974,0.111111


### Campaigns with the highest absolute lift

**Absolute lift** is the difference in conversion rate (treatment minus control). The table below shows the **top five** by lift—strong candidates to review for **scale** when incremental revenue also supports the story.

In [8]:
top_lift = lift.sort_values("absolute_lift", ascending=False, na_position="last").head(5)[cols]
top_lift

,campaign_id,incremental_revenue,absolute_lift,treatment_conversion_rate,control_conversion_rate
11,6,671.64,0.411765,0.411765,0.000000
9,4,329.56,0.333333,0.333333,0.000000
17,12,381.62,0.333333,0.333333,0.000000
3,15,458.26,0.299283,0.354839,0.055556
10,10,371.37,0.294737,0.400000,0.105263


### Inefficient campaigns (negative lift)

These campaigns had **lower conversion in treatment than control** during the test window. They warrant **review** (targeting, creative, landing experience, or test execution) and are natural candidates for **reduced or paused spend** once stakeholders agree the read is trusted.

In [9]:
bad = lift["absolute_lift"].notna() & (lift["absolute_lift"] < 0)
inefficient = lift.loc[bad].sort_values("absolute_lift", ascending=True)[
    ["campaign_id", "absolute_lift", "incremental_revenue", "treatment_conversion_rate", "control_conversion_rate"]
]
flags_dedup = flags.drop_duplicates(subset=["campaign_id"], keep="first")
inefficient = inefficient.merge(flags_dedup[["campaign_id", "efficiency_flag"]], on="campaign_id", how="left")
inefficient

,campaign_id,absolute_lift,incremental_revenue,treatment_conversion_rate,control_conversion_rate,efficiency_flag
0,14,-0.003003,-4.15,0.108108,0.111111,inefficient


### Rankings snapshot (combined score)

The **rankings** mart adds **revenue rank**, **lift rank**, and a **combined_score** (equal mix of normalized lift and incremental revenue). Below: **top five** by `combined_score`—useful for a **single prioritized list** in steering meetings.

In [10]:
rank_cols = [
    "campaign_id",
    "incremental_revenue",
    "absolute_lift",
    "incremental_revenue_rank",
    "absolute_lift_rank",
    "combined_score",
]
top_combo = rankings.sort_values("combined_score", ascending=False, na_position="last").head(5)[rank_cols]
top_combo

,campaign_id,incremental_revenue,absolute_lift,incremental_revenue_rank,absolute_lift_rank,combined_score
0,6,671.64,0.411765,1,1,1.000000
3,15,458.26,0.299283,4,4,0.706530
6,12,381.62,0.333333,7,3,0.690873
1,24,525.30,0.229167,2,8,0.671606
8,4,329.56,0.333333,9,2,0.652355


### Decision summary

- **Grow:** campaigns that show up in **top incremental revenue** and **strong lift** (and high **combined_score**) are the best starting points for **budget increases** or **expanded reach**, assuming operational constraints allow.
- **Maintain / monitor:** **moderate** lift or **low** incremental revenue with flat lift suggests **holding spend** and **monitoring** the next read—avoid large moves on noise.
- **Reduce / fix:** **Negative lift** campaigns should trigger **operational review** first; if the experiment is sound, **reduce budget** or **pause** until there is a clear fix (creative, audience, or execution).
- **One line for leadership:** incrementality tells you **where the experiment says the next dollar works hardest**—combine it with **margin, supply, and strategic priorities** before locking the plan.
